# Nexora Analytics
## Detección de ventas de alto valor y segmentación de clientes: Online Retail II

**Actividad:** Producto aplicado 2 · Pipeline de IA para predicción y segmentación
**Dataset:** Online Retail II (`Online_Retail_II_Depurado_App.xlsx`, hojas *Year 2009-2010* y *Year 2010-2011*)

## 1. Introducción y objetivo

Este cuaderno recorre de principio a fin un pipeline de aprendizaje automático
sobre el histórico de transacciones de Online Retail II, entre diciembre de 2009
y diciembre de 2011. La idea es simple de enunciar y algo más exigente de
sostener con datos: identificar qué ventas vale la pena vigilar de cerca, y
entender qué tipos de clientes hay detrás de esas ventas. Para eso, el cuaderno
pasa por:

1. **ETL:** cargar, limpiar y enriquecer los datos transaccionales.
2. **EDA:** conocer el dataset antes de intentar modelarlo.
3. **Modelo supervisado:** un clasificador de ventas de alto valor, puesto a
   prueba contra una línea base.
4. **Modelo no supervisado:** segmentación de clientes por Recencia, Frecuencia y
   Monto (RFM).
5. **Juicio profesional:** si estos modelos, tal como están, deberían pasar a
   producción o no.

Todos los importes están en libras esterlinas (GBP). Para correrlo, suba
`Online_Retail_II_Depurado_App.xlsx` al entorno de Colab y ejecute las celdas en
orden desde el principio.

De aquí salen dos archivos: `OnlineRetailII_limpio.csv` (el dataset ya depurado,
sección 2) y `OnlineRetailII_segmentos.csv` (un registro por cliente con el
segmento que le tocó, sección 5).

# 2. Carga y preparación de datos

## 2.1 Carga y consolidación

El archivo original reparte las transacciones en dos hojas de Excel, una por
rango de años. Lo primero es juntarlas en una sola tabla para trabajar con todo
el histórico de una vez.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ARCHIVO = "Online_Retail_II_Depurado_App.xlsx"
if not Path(ARCHIVO).exists():
    raise FileNotFoundError(f"No se encuentra '{ARCHIVO}'. Súbalo al entorno de Colab.")

xls = pd.ExcelFile(ARCHIVO)
d = pd.concat([xls.parse("Year 2009-2010"), xls.parse("Year 2010-2011")], ignore_index=True)

print("Registros consolidados:", f"{len(d):,}")
d.head(3)

## 2.2 Limpieza: duplicados, nulos, valores inválidos, cancelaciones

Empezamos quitando las filas que están duplicadas de forma exacta. El resto de
controles habituales (nulos, cantidades o precios que no tienen sentido,
facturas de cancelación con el prefijo "C") ya venían resueltos en el archivo
que recibimos, pero igual los verificamos aquí para dejar constancia de que
efectivamente no aparecen.

In [ ]:
antes = len(d)
d = d.drop_duplicates().reset_index(drop=True)

print(f"Registros antes de deduplicar : {antes:,}")
print(f"Registros después             : {len(d):,}")
print(f"Duplicados eliminados         : {antes-len(d):,} ({(antes-len(d))/antes*100:.2f} %)")
print()
print("Nulos por columna:")
print(d.isna().sum().to_string())
print()
print("Cantidad <= 0            :", (d["Quantity"] <= 0).sum())
print("Precio <= 0              :", (d["Price"] <= 0).sum())
print("Facturas de cancelación  :", d["Invoice"].astype(str).str.startswith("C").sum())

## 2.3 Variables derivadas

Renombramos las columnas al español y creamos `ingreso`, `anio` y `mes`.
También construimos una **categoría de producto aproximada**: el dataset no
trae ninguna clasificación oficial de producto ni de canal de venta, así que
buscamos palabras clave frecuentes en la descripción de cada artículo y
agrupamos a partir de ahí. Vale la pena ser honestos sobre lo que es esto: una
simplificación por coincidencia de texto, no una taxonomía validada del
catálogo. Su cobertura real y sus límites se cuantifican en la sección 3.4.

In [ ]:
d = d.rename(columns={
    "Invoice": "id_pedido", "Customer ID": "id_cliente", "Country": "pais",
    "Quantity": "cantidad", "Price": "precio_unitario",
    "InvoiceDate": "fecha", "Description": "producto"})

d["ingreso"] = d["cantidad"] * d["precio_unitario"]
d["anio"] = d["fecha"].dt.year
d["mes"] = d["fecha"].dt.month

PALABRAS_CLAVE = ["HEART", "BAG", "BOX", "CHRISTMAS", "LIGHT", "CARD",
                   "CANDLE", "BOTTLE", "MUG", "BUNTING"]
desc = d["producto"].astype(str).str.upper()
categoria = pd.Series("OTROS", index=d.index)
for palabra in PALABRAS_CLAVE:
    libre = categoria == "OTROS"
    categoria[libre & desc.str.contains(palabra, na=False)] = palabra
d["categoria"] = categoria

print(f"Registros    : {len(d):,}")
print(f"Clientes     : {d['id_cliente'].nunique():,}")
print(f"Pedidos      : {d['id_pedido'].nunique():,}")
print(f"Países       : {d['pais'].nunique()}")
print(f"Rango fechas : {d['fecha'].min().date()} a {d['fecha'].max().date()}")

## 2.4 Exportación del dataset depurado

In [ ]:
d.to_csv("OnlineRetailII_limpio.csv", index=False)
print("Exportado OnlineRetailII_limpio.csv:", f"{len(d):,}", "registros")

# 3. Análisis exploratorio de datos (EDA)

Antes de entrenar cualquier modelo conviene mirar los datos de frente. Esta
sección busca entender cómo se comporta el ingreso, cómo varían las ventas en
el tiempo, dónde están los clientes y qué tan bien queda cubierta la categoría
de producto que acabamos de construir.

## 3.1 Estadísticas generales del ingreso por línea de venta

In [ ]:
print(d["ingreso"].describe().round(2).to_string())
print()
print(f"Ingreso total del periodo : £ {d['ingreso'].sum():,.0f}")
print(f"Percentil 95              : £ {d['ingreso'].quantile(0.95):.2f}")
print(f"Percentil 99              : £ {d['ingreso'].quantile(0.99):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.6))
ax.hist(np.log1p(d["ingreso"]), bins=60, color="#2073F4", edgecolor="white")
ax.set_xlabel("log(1 + ingreso), ingreso en GBP")
ax.set_ylabel("Número de líneas de venta")
ax.set_title("Distribución del ingreso por línea de venta (escala log)", fontsize=13, weight="bold")
ax.grid(axis="y", color="#E3E9F1", lw=0.9)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

### Interpretación

**Qué muestra:** que el ingreso por línea de venta está lejos de ser simétrico.
La mediana apenas llega a £12,45 y la media ya sube a £21,98 (señal de que unos
pocos valores extremos están tirando del promedio hacia arriba), y el caso más
grande de todos es una sola línea de £168.469,60. En números simples: casi
todas las ventas son pequeñas, y unas pocas son enormes.

**Qué decisión cambiaría:** justamente por esa asimetría, en la sección 4 no
tendría sentido usar la media ni un corte arbitrario para definir qué es "alto
valor"; por eso se recurre a un criterio más robusto (1,5 × IQR). También
adelanta algo importante: la clase "alto valor" va a ser minoritaria, y eso va
a condicionar toda la evaluación del modelo más adelante.

**Qué limitación tiene:** el histograma está en escala logarítmica a propósito,
porque en escala normal esos valores extremos casi ni se verían junto al
montón de ventas pequeñas. Es una decisión de visualización, no un maquillaje
de los datos, pero conviene tenerlo presente al leer el eje.

## 3.2 Evolución mensual de ventas

In [ ]:
mensual = d.assign(periodo=d["fecha"].dt.to_period("M")).groupby("periodo")["ingreso"].sum()

fig, ax = plt.subplots(figsize=(10.5, 4.6))
ax.plot(mensual.index.astype(str), mensual.values, marker="o", color="#2073F4", lw=2)
ax.set_xlabel("Mes")
ax.set_ylabel("Ingreso total (GBP)")
ax.set_title("Ingreso mensual, 2009-2011", fontsize=13, weight="bold")
ax.tick_params(axis="x", rotation=90)
ax.grid(color="#E3E9F1", lw=0.9)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

### Interpretación

**Qué muestra:** que las ventas tienen una estacionalidad bastante marcada.
Octubre y noviembre se repiten como los meses fuertes en los dos años del
dataset (noviembre de 2010 y de 2011 superan ambos los £1,13 millones), y
después vienen diciembre y los primeros meses del año siguiente, más flojos.
Tiene sentido: son los meses previos a la temporada de fin de año.

**Qué decisión cambiaría:** si alguien está planificando inventario o campañas
comerciales, esta estacionalidad es motivo suficiente para incluir el mes como
variable del modelo (algo que ya se hace en la sección 4) y para pensar en
evaluar el modelo por separado en temporada alta y en temporada baja, en vez de
mezclarlo todo.

**Qué limitación tiene:** solo tenemos dos ciclos anuales completos. Eso
alcanza para decir que el patrón se repitió dos veces, no para asegurar que se
va a repetir siempre; con dos años de datos, cualquier afirmación sobre "todos
los años" sería una sobre-generalización.

## 3.3 Concentración geográfica

In [ ]:
top_pais = d.groupby("pais")["ingreso"].sum().sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(8.6, 4.8))
ax.barh(top_pais.index[::-1], top_pais.values[::-1], color="#2073F4")
ax.set_xscale("log")
ax.set_xlabel("Ingreso total (GBP, escala log)")
ax.set_title("Los 8 países con mayor ingreso", fontsize=13, weight="bold")
ax.grid(axis="x", color="#E3E9F1", lw=0.9, which="both")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

### Interpretación

**Qué muestra:** que esto, más que un negocio internacional, es un negocio
británico con clientes ocasionales en otros países. Reino Unido concentra
£14,29 millones de ingreso y 5.334 de los 5.852 clientes (el 91 %); el segundo
país en la lista, Irlanda, ni siquiera llega a £587 mil.

**Qué decisión cambiaría:** conviene tener claro que cualquier modelo o
segmentación entrenado sobre este dataset está describiendo, en el fondo, el
comportamiento del mercado británico. Extrapolar las conclusiones a otros
países hay que hacerlo con cuidado, y como variable del modelo, `pais` aporta
poco fuera del Reino Unido simplemente porque no hay suficientes datos de los
demás mercados.

**Qué limitación tiene:** el eje está en escala logarítmica porque, si no,
casi ningún país aparte de Reino Unido se vería en el gráfico. Es necesario
para poder comparar, pero distorsiona un poco la sensación visual de qué tan
lejos están unos países de otros.

## 3.4 Cobertura de la categoría de producto aproximada

In [ ]:
cobertura = d["categoria"].value_counts()

fig, ax = plt.subplots(figsize=(8.6, 4.8))
ax.bar(cobertura.index, cobertura.values, color="#2073F4")
ax.set_ylabel("Número de líneas de venta")
ax.set_title("Distribución de la categoría de producto aproximada", fontsize=13, weight="bold")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", color="#E3E9F1", lw=0.9)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Cobertura con categoría nombrada: {(d['categoria']!='OTROS').mean()*100:.1f} %")

### Interpretación

**Qué muestra:** que la categoría aproximada cubre menos de la mitad de los
datos. Solo el 40,4 % de las líneas de venta cae en alguna de las diez
palabras clave (HEART, BAG, BOX, CHRISTMAS, LIGHT, CARD, CANDLE, BOTTLE, MUG o
BUNTING); el 59,6 % restante queda simplemente como "OTROS".

**Qué decisión cambiaría:** con casi 60 % de los datos sin una categoría
específica, es de esperar que `categoria` le aporte al modelo de la sección 4
bastante menos de lo que podría. Si en algún momento se consigue una taxonomía
real de producto (por ejemplo, a partir del `StockCode`), vale la pena
reemplazar esta variable por esa, porque probablemente el modelo mejore.

**Qué limitación tiene:** esta categoría nace de buscar palabras dentro de un
texto libre, no de una clasificación real del catálogo. Dos productos pueden
compartir la palabra clave y no tener nada que ver entre sí, y al revés:
productos del mismo tipo pueden terminar repartidos en categorías distintas
simplemente porque su descripción usa otras palabras.

# 4. Modelo supervisado: detección de ventas de alto valor

## 4.1 Definición del objetivo

Antes de entrenar nada hay que resolver una pregunta que parece obvia y no lo
es: ¿qué significa exactamente "alto valor"? Aquí se define como toda venta
cuyo ingreso supera `Q3 + 1.5 × IQR`, el criterio habitual para marcar valores
atípicos en una distribución. Se prefirió este criterio sobre un umbral fijo
inventado a mano porque se adapta a cómo se comporta realmente el ingreso en
este dataset (sección 3.1), en vez de imponerle un corte arbitrario que no
tiene por qué tener sentido para estos datos en particular.

In [ ]:
q1 = d["ingreso"].quantile(0.25)
q3 = d["ingreso"].quantile(0.75)
limite = q3 + 1.5 * (q3 - q1)
d["alto_valor"] = (d["ingreso"] > limite).astype(int)

print(f"Umbral de alto valor  : £ {limite:,.2f}")
print(f"Líneas de venta       : {len(d):,}")
print(f"De alto valor         : {d['alto_valor'].sum():,}  ({d['alto_valor'].mean()*100:.2f} %)")

## 4.2 Variables predictoras

El modelo usa `categoria`, `pais`, `cantidad` y `mes`. La lógica detrás de esta
elección es simple: son variables que ya se conocen en el momento en que se
registra la venta, antes de que exista ninguna duda sobre si esa venta
terminará siendo de alto valor o no.

In [ ]:
X = pd.get_dummies(d[["categoria", "pais", "cantidad", "mes"]],
                    columns=["categoria", "pais"], dummy_na=True)
y = d["alto_valor"]

print("Variables de entrada:", X.shape[1])

## 4.3 Control de fuga de datos

`precio_unitario` e `ingreso` quedan fuera del modelo a propósito.
`ingreso = cantidad × precio_unitario`, y el objetivo se define justamente
sobre `ingreso`, así que darle cualquiera de las dos variables al modelo sería
como darle la respuesta del examen junto con la pregunta: aprendería a
despejar una ecuación, no a predecir nada. Y en un escenario real, esa
información tampoco estaría disponible de la misma forma al momento de
anticipar si una venta va a ser grande o no.

Para dejar esto demostrado y no solo dicho, se entrena un modelo aparte que sí
usa `cantidad` y `precio_unitario` como únicas variables. Que quede claro desde
ya: ese modelo es **únicamente un experimento de control**. No se recomienda
llevarlo a producción bajo ninguna circunstancia.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_fuga = d[["cantidad", "precio_unitario"]]
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(X_fuga, y, test_size=0.25, stratify=y, random_state=42)
arbol_fuga = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xf_tr, yf_tr)
acierto_fuga = accuracy_score(yf_te, arbol_fuga.predict(Xf_te))

print(f"Acierto del modelo de control (con precio_unitario): {acierto_fuga*100:.2f} %")

## 4.4 Train/Test

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

print("Entrenamiento:", f"{len(X_tr):,}")
print("Prueba       :", f"{len(X_te):,}")
print(f"Proporción de altos · entrenamiento: {y_tr.mean()*100:.2f} %")
print(f"Proporción de altos · prueba       : {y_te.mean()*100:.2f} %")

## 4.5 Línea base

In [ ]:
from sklearn.dummy import DummyClassifier

base = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
acierto_base = accuracy_score(y_te, base.predict(X_te))

print(f"Acierto de la línea base: {acierto_base*100:.2f} %")

## 4.6 Árbol de decisión

In [ ]:
arbol = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, random_state=42).fit(X_tr, y_tr)
pred = arbol.predict(X_te)
acierto = accuracy_score(y_te, pred)

print(f"Línea base : {acierto_base*100:.2f} %")
print(f"Modelo     : {acierto*100:.2f} %")
print(f"Mejora     : {(acierto-acierto_base)*100:+.2f} puntos")

## 4.7 Evaluación

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_te, pred, target_names=["No alto valor", "Alto valor"], zero_division=0))

### Interpretación

**Qué muestra:** el árbol llega a 94,70 % de acierto global, 2,7 puntos por
encima de la línea base (91,99 %). Pero hay que mirar más de cerca la clase
que de verdad importa: en "Alto valor", la precisión es del 80 % y el recall
del 45 %. Dicho de otra forma, de cada 100 ventas de alto valor que existen de
verdad, el modelo encuentra 45 y se le escapan 55.

**Qué decisión cambiaría:** un recall de 45 % quiere decir que más de la mitad
de las ventas grandes pasan sin ser detectadas. Si la idea es que el equipo
comercial reciba una alerta cuando ocurra una de estas ventas, este modelo tal
como está todavía no sirve para eso: haría falta ajustar el umbral de decisión
o balancear las clases antes de pensar en usarlo de verdad (la sección 6 entra
en detalle).

**Qué limitación tiene:** el 94,70 % de acierto global suena bien, pero por sí
solo no dice mucho. Como solo el 8 % de las ventas son de alto valor, un
modelo que ni siquiera intentara detectarlas ya acertaría el 92 % de las veces
con no hacer nada. El recall es la métrica que deja esto en evidencia; el
acierto global, solo, lo hubiera escondido.

## 4.8 Matriz de confusión

In [ ]:
from sklearn.metrics import confusion_matrix

mc = confusion_matrix(y_te, pred)
print(mc)

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 5.4))
ax.imshow(mc, cmap="Blues")
etiquetas = ["No alto valor", "Alto valor"]
ax.set_xticks([0, 1]); ax.set_xticklabels(etiquetas)
ax.set_yticks([0, 1]); ax.set_yticklabels(etiquetas)
ax.set_xlabel("Predicción del modelo")
ax.set_ylabel("Valor real")
nombres_celda = [["Verdaderos\nnegativos", "Falsos\npositivos"],
                 ["Falsos\nnegativos", "Verdaderos\npositivos"]]
for i in range(2):
    for j in range(2):
        color_texto = "white" if mc[i][j] > mc.max()/2 else "black"
        ax.text(j, i, f"{mc[i][j]:,}\n{nombres_celda[i][j]}", ha="center", va="center",
                fontsize=10.5, color=color_texto)
ax.set_title("Matriz de confusión", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()

### Interpretación

**Qué muestra:** cómo se reparten los aciertos y los errores, celda por celda:
176.786 verdaderos negativos, 1.814 falsos positivos, 8.484 falsos negativos y
7.065 verdaderos positivos.

**Qué decisión cambiaría:** el número que de verdad debería preocupar es el de
los falsos negativos: 8.484 ventas de alto valor que el modelo no detectó. Si
el objetivo del negocio es justamente no perderse esas ventas, ese es el error
a atacar, no el acierto global, que, como vimos, puede ser engañoso.

**Qué limitación tiene:** el problema es que bajar los falsos negativos casi
siempre sube los falsos positivos, o sea, más alertas que resultan ser falsas
alarmas. Esta matriz no dice cuál de los dos errores le cuesta más caro al
negocio; eso necesitaría ponerle un número al costo de cada tipo de error, algo
que este análisis todavía no hace.

## 4.9 Comparación contra la línea base y el control de fuga

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.8))
nombres = ["Línea base", "Modelo Nexora", "Control de fuga\n(referencia, no producción)"]
valores = [acierto_base*100, acierto*100, acierto_fuga*100]
colores = ["#5B6472", "#2073F4", "#C0392B"]

barras = ax.bar(nombres, valores, color=colores, width=0.55)
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width()/2, v + 1, f"{v:.1f} %", ha="center", fontsize=12, weight="bold")
ax.axhline(acierto_base*100, color="#5B6472", ls="--", lw=1.4)
ax.set_ylim(0, 105)
ax.set_ylabel("Acierto (%)")
ax.set_title("Desempeño del modelo de detección de ventas de alto valor", fontsize=13, weight="bold")
ax.grid(axis="y", color="#E3E9F1", lw=0.9)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

importancias = pd.Series(arbol.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importancias[importancias > 0].round(4).to_string())

### Interpretación

**Qué muestra:** tres números lado a lado cuentan la historia completa. El
modelo de producción (94,70 %) le gana claramente a la línea base (91,99 %), y
queda bastante por debajo del control de fuga (98,57 %), ese modelo que usa
`precio_unitario` y que, como se explicó en la sección 4.3, nunca debería
llegar a producción. Mirando qué variable usó más el árbol, `cantidad` se
lleva el 94 % de la importancia.

**Qué decisión cambiaría:** que el modelo le gane a la línea base es la razón
para seguir puliéndolo en vez de descartarlo; y que se mantenga lejos del
control de fuga confirma que no está haciendo trampa por otro camino, aunque
no use el precio directamente.

**Qué limitación tiene:** ojo con sobre-interpretar la importancia de
`cantidad`. Ese número dice cuánto usó el árbol esa variable para separar los
datos, no que comprar más unidades sea la causa de que una venta se vuelva de
alto valor. Es una asociación que el modelo encontró útil, no una explicación
de por qué pasa lo que pasa.

## 4.10 Limitaciones del modelo supervisado

Antes de pasar a la segmentación, vale la pena dejar escritas, sin adornos,
las cosas que este modelo todavía no resuelve:

- El recall de la clase "Alto valor" (45 %) se queda corto para cualquier caso
  de uso donde no detectar una venta grande salga caro.
- La categoría de producto es una aproximación por texto libre, con 40,4 % de
  cobertura (sección 3.4); una taxonomía real basada en `StockCode`
  probablemente le daría al modelo más información con la que trabajar.
- El modelo se evaluó sobre una partición al azar del mismo periodo
  (2009-2011); todavía no sabemos si generaliza bien a un periodo futuro que
  no haya visto.
- Nadie estimó todavía cuánto cuesta, en términos de negocio, un falso
  positivo frente a un falso negativo, así que el umbral de decisión se dejó
  en el valor por defecto (0,5), que no está pensado para minimizar ese costo
  en particular.

# 5. Modelo no supervisado: segmentación RFM

## 5.1 Construcción RFM

Cambiamos de pregunta. Ya no se trata de predecir una venta puntual, sino de
entender qué tipos de clientes hay detrás de todas esas ventas. Para eso se
calculan tres variables por cliente, el enfoque clásico de RFM:

- **Recencia:** cuántos días pasaron desde su última compra hasta la fecha más
  reciente del dataset.
- **Frecuencia:** en cuántos pedidos (`id_pedido`) distintos compró.
- **Monto:** cuánto sumó en total durante todo el periodo.

In [ ]:
referencia = d["fecha"].max()

perfil = d.groupby("id_cliente").agg(
    recencia=("fecha", lambda s: (referencia - s.max()).days),
    frecuencia=("id_pedido", "nunique"),
    monto=("ingreso", "sum")
).reset_index()

print("Clientes                       :", f"{len(perfil):,}")
print("Con monto acumulado <= 0       :", (perfil["monto"] <= 0).sum())
print()
print(perfil[["recencia", "frecuencia", "monto"]].describe().round(1).to_string())

## 5.2 Transformación y escalado

Igual que pasaba con el ingreso en la sección 3.1, `frecuencia` y `monto`
están lejos de ser simétricas: hay clientes que compran muchísimo más que el
resto. Por eso se aplica `log1p` antes de estandarizar; si no, esos pocos
clientes extremos terminarían dominando por completo las distancias que usa
K-Means, y el resto de la base quedaría aplastado en un solo punto.

In [ ]:
from sklearn.preprocessing import StandardScaler

perfil["log_frecuencia"] = np.log1p(perfil["frecuencia"])
perfil["log_monto"] = np.log1p(perfil["monto"])

Z = StandardScaler().fit_transform(perfil[["recencia", "log_frecuencia", "log_monto"]])
print("Matriz escalada:", Z.shape)

## 5.3 Selección de K

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inercia, silueta = [], []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Z)
    inercia.append(km.inertia_)
    silueta.append(silhouette_score(Z, km.labels_))

for k, i, s in zip(range(2, 9), inercia, silueta):
    print(f"k = {k}   inercia = {i:8.1f}   silueta = {s:.4f}")

Aquí conviene detenerse un momento, porque el número que da la métrica no
siempre es el número correcto para el negocio. La silueta más alta se obtiene
con k = 2 (0,418), pero ese resultado separa a apenas 23 clientes de gasto
extremo del resto de toda la base: matemáticamente es lo mejor, pero en la
práctica no sirve para diseñar acciones comerciales distintas por grupo. Con
k = 3 la silueta baja muy poco (0,402) y a cambio aparecen tres segmentos con
perfiles bien diferenciados entre sí (sección 5.6). Por eso se fija **K = 3**:
es una decisión de negocio, no el resultado de perseguir el número más alto.

## 5.4 K-Means

In [ ]:
K = 3
km = KMeans(n_clusters=K, random_state=42, n_init=10).fit(Z)
perfil["grupo"] = km.labels_

print(f"K seleccionado: {K}")

## 5.5 Silhouette Score

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.6, 4.6))
a1.plot(range(2, 9), inercia, marker="o", color="#2073F4", lw=2.2)
a1.set_xlabel("Número de segmentos (k)"); a1.set_ylabel("Inercia")
a1.set_title("Método del codo", fontsize=12.5, weight="bold")
a2.plot(range(2, 9), silueta, marker="s", color="#1FA890", lw=2.2)
a2.axvline(K, color="#E8A33D", ls="--", lw=1.8)
a2.set_xlabel("Número de segmentos (k)"); a2.set_ylabel("Silueta media")
a2.set_title(f"Silueta · k seleccionado = {K}", fontsize=12.5, weight="bold")
for a in (a1, a2):
    a.grid(color="#E3E9F1", lw=0.9)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

### Interpretación

**Qué muestra:** cómo se mueven la inercia y la silueta media a medida que se
prueban entre 2 y 8 segmentos; la línea vertical marca el k = 3 que finalmente
se eligió.

**Qué decisión cambiaría:** de haberse dejado llevar únicamente por la silueta
más alta (k = 2, con 0,418), la segmentación habría terminado siendo "23
clientes extremos contra todos los demás", que no ayuda en nada a diseñar
acciones comerciales distintas por grupo. Al elegir k = 3 se resigna apenas
0,016 de silueta, y a cambio se ganan segmentos que sí se pueden usar
(sección 5.6).

**Qué limitación tiene:** conviene no engañarse con el número: una silueta de
0,402 es moderada, no alta. Los grupos se tocan entre sí y no son
compartimentos perfectamente separados. Esta segmentación sirve para priorizar
dónde poner el esfuerzo comercial, no para asumir que la pertenencia de un
cliente a un grupo es un hecho grabado en piedra.

## 5.6 Segmentos

In [ ]:
resumen = perfil.groupby("grupo").agg(
    clientes=("id_cliente", "count"),
    recencia=("recencia", "mean"),
    frecuencia=("frecuencia", "mean"),
    monto=("monto", "mean")).round(1)
print(resumen.to_string())

**Segmento 0: Clientes inactivos** · 1.823 clientes · monto medio £542
Su última compra fue hace 470 días en promedio, y tampoco compraban con mucha
frecuencia antes de eso. No tiene mucho sentido incluirlos en campañas activas
de venta cruzada; si se decide intentar recuperarlos, necesitan una campaña de
reenganche propia, distinta de cualquier programa de fidelización.

**Segmento 1: Clientes de alto valor** · 1.675 clientes · monto medio £8.420
Compran seguido (15,8 pedidos en promedio) y hace poco (57 días). Son apenas
el 29 % de la base, pero son, con diferencia, el segmento más rentable.
Ameritan atención dedicada y prioridad en cualquier programa de fidelización
que se arme.

**Segmento 2: Clientes activos regulares** · 2.354 clientes · monto medio £840
Su recencia es moderada (90 días) y compran poco (2,9 pedidos). Es el
segmento con más margen para crecer: venta cruzada e incentivos a la recompra
tienen sentido aquí.

Una aclaración importante: estos nombres describen un comportamiento histórico
de compra, no son etiquetas permanentes. Un cliente puede cambiar de segmento
si cambia su forma de comprar. Tampoco se encontraron clientes con monto
acumulado igual o menor a cero en estos datos.

In [ ]:
from sklearn.metrics import adjusted_rand_score

km2 = KMeans(n_clusters=K, random_state=7, n_init=10).fit(Z)
ari = adjusted_rand_score(km.labels_, km2.labels_)
print(f"Estabilidad frente a otra semilla (ARI): {ari:.4f}")

Un ARI de 0,993 quiere decir que, aunque se cambie la semilla aleatoria,
K-Means termina agrupando a los clientes casi exactamente igual. Eso da
tranquilidad: la partición no es un capricho de cómo se inicializó el
algoritmo, es consistente. Pero hay que tener cuidado de no leer más de la
cuenta en este número: que el algoritmo sea estable no significa que estos
tres perfiles vayan a seguir siendo válidos para siempre. La estabilidad es
del algoritmo, no una garantía sobre cómo se va a comportar el negocio de acá
en adelante.

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 5.6))
paleta = ["#2073F4", "#1FA890", "#E8A33D"]
nombres_grupo = {0: "Inactivos", 1: "Alto valor", 2: "Activos regulares"}
for g in sorted(perfil["grupo"].unique()):
    sub = perfil[perfil["grupo"] == g]
    ax.scatter(sub["frecuencia"], sub["monto"], s=34, alpha=0.65, color=paleta[g % len(paleta)],
               label=f"{nombres_grupo.get(g, g)} · {len(sub)} clientes")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Pedidos distintos por cliente (escala log)")
ax.set_ylabel("Monto acumulado en GBP (escala log)")
ax.set_title("Segmentos de clientes", fontsize=13, weight="bold")
ax.legend(frameon=False)
ax.grid(color="#EEF2F7", lw=0.9, which="both")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

### Interpretación

**Qué muestra:** dónde cae cada cliente según cuánto compra y cuánto gasta, en
escala logarítmica, coloreado según su segmento.

**Qué decisión cambiaría:** el segmento "Alto valor" se ve claramente corrido
hacia frecuencia y monto altos, separado del resto. Eso respalda tratarlo
distinto (atención dedicada, fidelización) en vez de meterlo en las mismas
campañas masivas que se le mandan a todos los demás.

**Qué limitación tiene:** entre "Inactivos" y "Activos regulares" sí hay
bastante solape visual en la zona de baja frecuencia, lo cual es coherente con
que la silueta no sea muy alta. La frontera entre esos dos grupos es bastante
más débil que la que separa a "Alto valor" del resto, así que no convendría
usarla para tomar decisiones sobre un cliente individual sin revisar el caso
primero.

## 5.9 Exportación y limitaciones

In [ ]:
esperado = {"clientes": d["id_cliente"].nunique(), "grupos": K}
obtenido = {"clientes": len(perfil), "grupos": perfil["grupo"].nunique()}
for k in esperado:
    ok = "OK " if obtenido[k] == esperado[k] else "REVISAR"
    print(f"{ok} {k:10s} esperado {esperado[k]:>6}  obtenido {obtenido[k]:>6}")

perfil.to_csv("OnlineRetailII_segmentos.csv", index=False, encoding="utf-8-sig")
print("\nExportado OnlineRetailII_segmentos.csv")

**Limitaciones de la segmentación:** conviene recordar qué es exactamente
este perfil RFM y qué no es. Es una fotografía del comportamiento acumulado
hasta la fecha más reciente del dataset, no una predicción de lo que un
cliente va a hacer de ahora en adelante. Tampoco dice nada sobre qué compra
cada cliente (eso quedó fuera), solo cuánto, cuándo y con qué frecuencia. Y
como ya se comentó en la sección 5.3, una silueta de 0,402 describe grupos con
límites razonablemente definidos, no clústeres perfectamente separados entre
sí.

# 6. Juicio profesional: ¿producción o no?

**Modelo de detección de ventas de alto valor: no, todavía no.**

Vamos a las cifras: 94,70 % de accuracy (contra 91,99 % de la línea base), 80 %
de precisión y 45 % de recall en la clase "Alto valor", con 8.484 falsos
negativos y 1.814 falsos positivos sobre 194.149 casos de prueba. Un recall de
45 % es el problema central: si la idea es que el modelo avise al equipo
comercial cuando aparezca una venta grande, se le va a escapar más de la mitad
de las veces que debería avisar. Eso no es un detalle menor, es la función
principal del modelo fallando la mayor parte del tiempo. Para que la respuesta
pudiera ser afirmativa, haría falta al menos:

1. Una categoría de producto real basada en `StockCode`, en vez de las
   palabras clave sobre texto libre que se usan ahora (que hoy cubren apenas
   40,4 % de los datos).
2. Alguna técnica de balanceo de clases (`class_weight`, sobremuestreo),
   porque solo el 8 % de las ventas son de alto valor y el modelo lo nota.
3. Ponerle un número al costo de cada tipo de error (falso positivo contra
   falso negativo) para fijar el umbral de decisión con criterio de negocio,
   en vez de dejarlo en el 0,5 por defecto que trae la librería.
4. Una validación temporal de verdad: entrenar con 2009-2010 y probar sobre
   2011, para saber si el modelo generaliza a un periodo que nunca vio, no
   solo a una muestra al azar del mismo periodo que ya conoce.
5. Probar modelos con más capacidad (bosques aleatorios, *gradient boosting*)
   antes de asumir que 45 % de recall es lo máximo que se puede conseguir
   aquí.

**Segmentación de clientes: sí, con revisión periódica.**

Las cifras acompañan: silueta de 0,402 en k = 3 (moderada, no hay que
exagerarla), estabilidad de 0,993 en el ARI frente a un cambio de semilla, y
una diferencia de casi diez veces en el monto medio entre el segmento "Alto
valor" (£8.420) y los otros dos (£542 y £840). Esa diferencia es lo bastante
grande como para justificar tratar distinto a cada segmento, incluso con una
silueta que no es espectacular. A favor: es estable, es accionable, y está
construida sobre tres variables de negocio que cualquiera puede entender sin
ser científico de datos. En contra: los límites entre "Inactivos" y "Activos
regulares" son difusos (sección 5.8), no toma en cuenta qué compra cada
cliente, y describe el pasado, no predice el futuro. Antes de instalarla como
parte del proceso comercial, conviene revalidarla cada trimestre y, si en
algún momento aparecen más variables útiles (categoría preferida, país),
revisar si valdría la pena sumarlas.